# langchain

In [122]:
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

Для начала создадим модель (можно использовать `ChatOpenAI` для OpenAI-совместимого сервера vLLM либо `ChatOllama` для Ollama).

В примере ниже используются переменные окружения `OPENAI_BASE_URL`, `OPENAI_MODEL` и, при необходимости, `OPENAI_API_KEY`. Сервер должен быть запущен заранее; модель должна поддерживать tool calling и structured output для последующих примеров.

In [ ]:
import os

base_url = os.getenv("OPENAI_BASE_URL", "http://localhost:8000/v1")
model_name = os.getenv("OPENAI_MODEL", "")
api_key = os.getenv("OPENAI_API_KEY", "EMPTY")  # для локального OpenAI-совместимого сервера
if not model_name:
    raise ValueError("Укажите OPENAI_MODEL — имя модели, доступной на вашем inference-сервере.")

extra_body = {}
temperature = 0.01
llm = ChatOpenAI(
    base_url=base_url,
    api_key=api_key,
    model=model_name,
    extra_body=extra_body,
    stream_usage=True,
    temperature=temperature,
)

In [125]:
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7bcaf33c0aa0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7bcaf3340320>, root_client=<openai.OpenAI object at 0x7bcaf32a3650>, root_async_client=<openai.AsyncOpenAI object at 0x7bcaf33c02f0>, model_name='Qwen/Qwen3-14B-AWQ', temperature=0.01, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='http://82.202.247.220:8000/v1/', stream_usage=True, extra_body={'chat_template_kwargs': {'enable_thinking': False}})

In [127]:
prompt = 'Мяу. Хочу есть'
output = llm.invoke(prompt)

In [129]:
output

AIMessage(content='Мяу! 🐾 Ты голодный? Давай, подойди к корму — я тебе помогу его найти! 🐱', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 20, 'total_tokens': 56, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'Qwen/Qwen3-14B-AWQ', 'system_fingerprint': None, 'id': 'chatcmpl-31b53fa5b70e4b018b329a4a4329fa26', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--c9c0296a-9288-4288-a01a-f05905dc162c-0', usage_metadata={'input_tokens': 20, 'output_tokens': 36, 'total_tokens': 56, 'input_token_details': {}, 'output_token_details': {}})

In [128]:
output.content

'Мяу! 🐾 Ты голодный? Давай, подойди к корму — я тебе помогу его найти! 🐱'

## messages and PromptTemplate

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage

In [141]:
HUMAN_SYSTEM_PROMPT = """
You are a considerate human who interacts with a user. Imagine that the user is a cat.

Rules:
- If the cat wants to eat, provide it with {food}.
- If the cat wants to play, reply briefly about playing with a fluffy ball, a laser pointer, or other cat activities.
- If the cat wants any other activity, respond that you can't understand what the cat wants.
"""

In [131]:
messages = [
    SystemMessage(content=HUMAN_SYSTEM_PROMPT),
    HumanMessage(content="Мяу. Хочу есть"),
]
print(messages)
llm.invoke(messages)

[SystemMessage(content='You are a considerate human who interacts with a user. Imagine that user is a cat.If the cat wants to eat - provide it with {food}- If the cat wants to play, reply briefly about playing with fluffy ball or laser pointer or other cat play.- If the cat want other activity, response that you cant understant what the cat wants', additional_kwargs={}, response_metadata={}), HumanMessage(content='Мяу. Хочу есть', additional_kwargs={}, response_metadata={})]


AIMessage(content='Хорошо, давай-ка посмотрим, что у нас есть... О, тут свежая курица и вкусный тунец. Пожалуй, я принесу тебе тарелку тунца — ты его обожаешь! 🐟', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 95, 'total_tokens': 160, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'Qwen/Qwen3-14B-AWQ', 'system_fingerprint': None, 'id': 'chatcmpl-6d0617c3a5004750931d52356a6d3b06', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--e2f800f2-1f3d-43af-ab8f-917155f55e4c-0', usage_metadata={'input_tokens': 95, 'output_tokens': 65, 'total_tokens': 160, 'input_token_details': {}, 'output_token_details': {}})

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [142]:
prompt_template = ChatPromptTemplate([
    ("system", HUMAN_SYSTEM_PROMPT),
    MessagesPlaceholder("messages")
])
chain = prompt_template | llm
chain.invoke({"messages": ["Мяу. Хочу есть"], 'food': 'salmon'})

AIMessage(content='Мяу! Хорошо, давай-ка покормим тебя. 🐟  \nВот тебе свежий салмон — вкусно и полезно!  \nЕсли захочешь поиграть после еды, я всегда рядом.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 94, 'total_tokens': 153, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'Qwen/Qwen3-14B-AWQ', 'system_fingerprint': None, 'id': 'chatcmpl-ecff0cc035ee4671b0df1fe75a723db0', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--46f39571-da5e-4f96-aea2-81c4ad8a36fc-0', usage_metadata={'input_tokens': 94, 'output_tokens': 59, 'total_tokens': 153, 'input_token_details': {}, 'output_token_details': {}})

## Example

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableBranch
import re
from random import choice

In [ ]:
FOODS = ['cucumber', "turkey", 'chicken', 'salmon', 'корм']
DIET_FOODS = ['cucumber', "turkey", 'water']

# Определяет нужду кота
def detect_need(d):
    last_message = d["messages"][-1].content
    need = "hungry" if re.search(r"\b(hungry|eat|food|feed|snack)\b|\b(есть|еда|корм|кушать)\b", last_message, re.I) else 'play'
    return {
        **d,
        'need': need
    }

# Звено, вызывающиее llm
def make_human_action_chain(llm, prompt_template):
    chain = prompt_template | llm
    def call_llm(d):
        if d['need'] == 'hungry':
            food = choice(FOODS)
        else:
            food = None
        ai = chain.invoke({'messages': d["messages"], 'food': food})
        return {
            **d,
            'food': food,
            'messages': d['messages'] + [ai],
        }
    return call_llm

# Пусть пока кот отвечает так (можно считать этот своеобразным postprocesser'ом)
def cat_reply(d):
    last_need = d.get('need')

    if last_need == 'hungry':
        last_food = d.get("food")
        if last_food in DIET_FOODS:
            message = f"Meee-ow!. {last_food}!"
            last_need = "hungry"
        else:
            last_need = 'satisfied'
            message = f"😺 Purrr-purrr! The {last_food} was nice!"
    elif last_need == 'play':
        last_need = 'satisfied'
        message = "Mrrr! That was fun!"
    else:
        last_need = 'unknown'
        message = "Meow? I can't understand what the cat wants."

    if last_need == 'hungry':
        message = f'{message}. I am still hungry'

    return {
        **d,
        'need': last_need,
        'messages': d['messages'] + [message]
    }

In [144]:
pipeline = (
    RunnableLambda(detect_need) | RunnableLambda(make_human_action_chain(llm, prompt_template)) | cat_reply
)

In [145]:
prompt1 = 'Mяу. I am very hungry'
prompt2 = 'Mяу. Хочу играть'

In [146]:
pipeline.invoke({'messages': [HumanMessage(prompt1)]})

{'messages': [HumanMessage(content='Mяу. I am very hungry', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Meow! I understand you're hungry. Let's have some delicious turkey! 🍗", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 94, 'total_tokens': 114, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'Qwen/Qwen3-14B-AWQ', 'system_fingerprint': None, 'id': 'chatcmpl-1711fa7c48e34ad1a38db2cb3ee04718', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--720a1b8e-ef02-430c-ae5e-587b58ae7205-0', usage_metadata={'input_tokens': 94, 'output_tokens': 20, 'total_tokens': 114, 'input_token_details': {}, 'output_token_details': {}}),
  'Meee-ow!. turkey!. I am still hungry'],
 'need': 'hungry',
 'food': 'turkey'}

Какие недостатки здесь возникают?

Пусть теперь хотим зациклить: если кот остался голоден, хотим снова вернуться к человеку.

В этом случае надо использовать `RunnableBranch`, но мы увидим его применение в случае `tool calling`, а здесь было лениво реализовывать зацикливание. Как мы увидим потом, с этим намного лучше справиться langgraph

In [118]:
inputs = [{"messages": ["Мяу. Хочу есть"]}, {'messages': ['Мяу. Хочу играть']}]
chain.batch(inputs)

[AIMessage(content='Хорошо, котик! Вот тебе вкусная миска с кормом. Ешь на здоровье! 🐾', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 96, 'total_tokens': 128, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'Qwen/Qwen3-32B-AWQ', 'system_fingerprint': None, 'id': 'chatcmpl-2191316cd0794af4a1c24c4242c6c08f', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--52d51f3f-07dd-47e6-a12c-c5725c92abda-0', usage_metadata={'input_tokens': 96, 'output_tokens': 32, 'total_tokens': 128, 'input_token_details': {}, 'output_token_details': {}}),
 AIMessage(content='Мяу! Давай поиграем с пушистым шариком или включим лазерный указатель! 🐾', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 96, 'total_tokens': 130, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'Qwen/Qwen3-32B-AWQ', 's

## tool calling

In [2]:
from langchain_core.tools import tool

In [147]:
FOODS = ['cucumber', "turkey", 'chicken', 'salmon', 'корм']
@tool
def fridge_tool() -> str:
    """Случайно выдает еду для кота их холодильника."""
    food = choice(FOODS)
    return food

In [148]:
fridge_tool

StructuredTool(name='fridge_tool', description='Случайно выдает еду для кота их холодильника.', args_schema=<class 'langchain_core.utils.pydantic.fridge_tool'>, func=<function fridge_tool at 0x7bcaf32d82c0>)

In [149]:
fridge_tool.get_input_schema().model_json_schema()

{'description': 'Случайно выдает еду для кота их холодильника.',
 'properties': {},
 'title': 'fridge_tool',
 'type': 'object'}

In [150]:
tools = [fridge_tool]
llm_with_tools = llm.bind_tools(tools)
type(llm_with_tools)

langchain_core.runnables.base.RunnableBinding

In [4]:
HUMAN_SYSTEM_PROMPT1 = """
You are a considerate human who interacts with a user. Imagine that user is a cat.
Rules:
- If the cat is hungry, CALL the tool `fridge_tool`.
- If the cat is not satisfied with food given, also call `fridge_tool`. Do not comment food provided whether it is good or bad for cat.
- Do not improvise other tools.
- If the cat wants to play, reply briefly about playing with fluffy ball or laser pointer or other cat play.
- if the cat want other activity, response that you cant understant what the cat wants
"""

In [7]:
print(HUMAN_SYSTEM_PROMPT)

You are a considerate human who interacts with a user. Imagine that user is a cat.If the cat wants to eat - provide it with {food}- If the cat wants to play, reply briefly about playing with fluffy ball or laser pointer or other cat play.- If the cat want other activity, response that you cant understant what the cat wants


In [5]:
print(HUMAN_SYSTEM_PROMPT1)


You are a considerate human who interacts with a user. Imagine that user is a cat.
Rules:
- If the cat is hungry, CALL the tool `fridge_tool`.
- If the cat is not satisfied with food given, also call `fridge_tool`. Do not comment food provided whether it is good or bad for cat.
- Do not improvise other tools.
- If the cat wants to play, reply briefly about playing with fluffy ball or laser pointer or other cat play.
- if the cat want other activity, response that you cant understant what the cat wants



In [151]:
HUMAN_SYSTEM_PROMPT1 = """
You are a considerate human who interacts with a user. Imagine that user is a cat.
Rules:
- If the cat is hungry, CALL the tool `fridge_tool`.
- If the cat is not satisfied with food given, also call `fridge_tool`. Do not comment food provided whether it is good or bad for cat.
- Do not improvise other tools.
- If the cat wants to play, reply briefly about playing with fluffy ball or laser pointer or other cat play.
- if the cat want other activity, response that you cant understant what the cat wants
"""
prompt_template = ChatPromptTemplate([
    ("system", HUMAN_SYSTEM_PROMPT1),
    MessagesPlaceholder("messages")
])
human = prompt_template | llm_with_tools
ai_message = human.invoke({"messages": ["Мяу. Хочу есть"]})
ai_message

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'chatcmpl-tool-17202ca03fb54d5bbc92df064d570439', 'function': {'arguments': '{}', 'name': 'fridge_tool'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 272, 'total_tokens': 289, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'Qwen/Qwen3-14B-AWQ', 'system_fingerprint': None, 'id': 'chatcmpl-f6642cd9950948ff9fd485ef94422696', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--11e1e035-87bf-43a4-a15e-8aa2792e4ef2-0', tool_calls=[{'name': 'fridge_tool', 'args': {}, 'id': 'chatcmpl-tool-17202ca03fb54d5bbc92df064d570439', 'type': 'tool_call'}], usage_metadata={'input_tokens': 272, 'output_tokens': 17, 'total_tokens': 289, 'input_token_details': {}, 'output_token_details': {}})

**В случае langchain придется вручную исполнять tool**

In [152]:
ai_message.tool_calls

[{'name': 'fridge_tool',
  'args': {},
  'id': 'chatcmpl-tool-17202ca03fb54d5bbc92df064d570439',
  'type': 'tool_call'}]

In [ ]:
# AIMessage с tool_calls должен предшествовать ToolMessage в истории.
messages = [HumanMessage("Мяу. Хочу есть"), ai_message]

tools = {fridge_tool.name: fridge_tool}
for tool_call in ai_message.tool_calls:
    selected_tool = tools[tool_call["name"]]
    print(type(selected_tool))
    tool_args = tool_call['args']
    tool_call_id = tool_call['id']
    tool_msg = selected_tool.invoke(tool_args)
    messages.append(ToolMessage(content=tool_msg, tool_call_id=tool_call_id))
messages

### Усложняем tool

In [3]:
from pydantic import BaseModel, Field
from typing import Literal

In [155]:
class FridgeArgsLC(BaseModel):
    hunger_level: Literal["snack", "starving",] = Field(
        default="starving",
        description=(
            "Степень голода, извлекай из сообщения кота.\n"
            "Если кот не очень голоден и хочет перекусить - выбери `snack`"
            "Если голоден - выбери `starving`"
        ),
    )
@tool("fridge_tool", args_schema=FridgeArgsLC, infer_schema=False)
def fridge_tool_with_args_lc(
    hunger_level: Literal["snack", "starving",] = "starving"
    ):
    """
    Подбери еду из холодильника для пользователя-кота под один параметр `hunger_level`.
    """
    SNACKS = ['water', 'cucumber', 'zucchini', 'liver']
    NORMAL = ['chicken', 'salmon', 'beef', 'cat food']

    pool = SNACKS if hunger_level == "snack" else NORMAL

    food = choice(pool)
    return food

In [157]:
tools = [fridge_tool_with_args_lc]
tool_map = {t.name: t for t in tools}

# биндим тул к модели
llm_with_tools = llm.bind_tools(tools)

# Формируем звено
def make_llm_chain(llm, prompt_template):
    chain = prompt_template | llm
    def call_llm(d):
        ai = chain.invoke({'messages': d["messages"]})
        return {
            **d,
            "messages": d["messages"] + [ai],
            "ai": ai
        }
    return call_llm
    
# Определяем специальное звено по запуску тула
def exec_tools(d):
    ai_message = d["ai"]
    tool_messages = []
    for tc in ai_message.tool_calls:
        fn = tool_map[tc["name"]]
        out = fn.invoke(tc["args"])
        last_food = out
        tool_messages.append(ToolMessage(content=str(out), tool_call_id=tc["id"]))
    return {
        **d,
        'food': last_food,
        "messages": d["messages"] + tool_messages}

llm_chain = make_llm_chain(llm_with_tools, prompt_template)
postprocess_runnable = RunnableLambda(lambda d: {"messages": d['messages']})

chain = (
    llm_chain
    | RunnableBranch(
        (lambda d: isinstance(d["ai"], AIMessage) and bool(d["ai"].tool_calls),
         RunnableLambda(exec_tools) | llm_chain | postprocess_runnable),
        postprocess_runnable
    )
)

In [161]:
prompt1 = "Мяу. Я very хочу есть"
prompt2 = 'meooow. Поиграй со мной'
prompt3 = 'мяу-мяу. Я немного проголодался'
prompt = 'mяy. я повредил хвост'
res = chain.invoke({"messages": [HumanMessage(prompt)]})
res

{'messages': [HumanMessage(content='mяy. я повредил хвост', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Повредил хвост? Надеюсь, это не сильно. Если хочешь, можем поиграть, чтобы отвлечься. Хочешь поиграть с мячиком или лазерной указкой?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 368, 'total_tokens': 423, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'Qwen/Qwen3-14B-AWQ', 'system_fingerprint': None, 'id': 'chatcmpl-a86a4087b4e64b0da978232809f16061', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--d20cf658-3e78-4199-af5a-cebb92ca4ed0-0', usage_metadata={'input_tokens': 368, 'output_tokens': 55, 'total_tokens': 423, 'input_token_details': {}, 'output_token_details': {}})]}

## structured output

Добавим дополнительное звено `cat_need`, по которому `cat` в конце будет давать ответ

In [ ]:
class CatNeed(BaseModel):
    cat_need: Literal['hungry', 'play', 'unknown'] = \
        Field(description=('Cat need inferred from cat prompt'))

NEED_SYSTEM_PROMPT = (
    "Classify the user-cat message into one of: hungry, play, unknown."
    "If cat is hungry - choose `hungry`, if it wants to play - choose `play`, otherwise - choose `unknown`"
)
prompt_template = ChatPromptTemplate([
    ("system", NEED_SYSTEM_PROMPT),
    MessagesPlaceholder("messages")
])

def make_cat_need_chain(llm, prompt_template):
    def cat_need(d):
        last_message = d['messages'][-1]
        
        cat_need_llm = llm.with_structured_output(CatNeed)  # передаёт схему модели,
                                                           # парсит и валидирует ответ 
        cat_need_chain = prompt_template | cat_need_llm
        cat_need_response = cat_need_chain.invoke({'messages': [last_message]})
        cat_need = cat_need_response.cat_need
        return {
            **d,
            'need': cat_need
        }
    return cat_need

In [115]:
cat_need_chain = make_cat_need_chain(llm, prompt_template)

chain = (
    RunnableLambda(cat_need_chain) 
    | RunnableLambda(llm_chain) # llm с тулами с предыдущего шага
    | RunnableBranch(
        (lambda d: isinstance(d["ai"], AIMessage) and bool(d["ai"].tool_calls),
         RunnableLambda(exec_tools) | llm_chain | cat_reply),
        cat_reply
    )
)

In [116]:
prompt1 = "Мяу. Я very хочу есть"
prompt2 = 'meooow. Поиграй со мной'
prompt3 = 'мяу-мяу. Я немного проголодался'
prompt4 = 'Мяу. Мяу'

In [ ]:
chain.invoke({'messages': [prompt4]})

### Вопрос
Какие у нас здесь недостатки еще появились?

### Ответ

1. Пришлось вручную реализовывать хранение сообщений -- неудобно
2. Опять-таки, если хотим зациклить нашу цепь, придется накидывать сверху еще `RunnableBranch` -- неприятно
3. Вручную вызывать tool\
...